# Batch Scene Alignment With LightGlue

This notebook reads `img_labels.csv`, groups images by `location`, aligns every image in each group to one anchor with SuperPoint + LightGlue, crops the shared overlap, saves cropped aligned JPEGs, and writes a results CSV.

In [ ]:
# Run once per Colab/runtime.
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

!pip install -q uv
!uv pip install --system git+https://github.com/cvg/LightGlue.git pillow pillow-heif pandas opencv-python-headless torch torchvision matplotlib tqdm

In [ ]:
import gc
import json
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import lightglue
from lightglue import LightGlue, SuperPoint
from lightglue.utils import load_image
from tqdm.auto import tqdm

plt.rcParams['figure.figsize'] = (14, 7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('LightGlue package:', lightglue.__file__)

def bgr_to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def safe_name(path):
    return Path(str(path)).with_suffix('').as_posix().replace('/', '__').replace(' ', '_')

In [ ]:
# Paths and batch settings. Edit these if your Drive layout differs.
BASE_CANDIDATES = [
    '/content/drive/MyDrive/CIS_5190_group_project',
    '/content/drive/My Drive/CIS_5190_group_project',
    os.path.abspath('../data'),
]
BASE = next((p for p in BASE_CANDIDATES if os.path.exists(p)), BASE_CANDIDATES[0])
PROCESSED_DIR = os.path.join(BASE, 'processedImages') if 'content' in BASE else os.path.join(BASE, 'processed')
LABELS_CSV = os.path.join(BASE, 'img_labels.csv')
OUTPUT_DIR = os.path.join(BASE, 'aligned_lightglue')
OUTPUT_CSV = os.path.join(BASE, 'aligned_lightglue_labels.csv')

# Use a list like ['34th', 'agh3rd'] while debugging, or None for every location.
LOCATION_FILTER = None

# LightGlue / homography settings.
RESIZE = 1024
MAX_KEYPOINTS = 2048
RANSAC_REPROJ_THRESHOLD = 5.0
MIN_MATCHES = 12
MIN_INLIERS = 8
# LightGlue can return hundreds of tentative matches, so the inlier ratio can be low
# even when the homography is visually good. Keep this at 0.0 to match the single-pair
# debug workflow, or raise it later if you see bad warps being accepted.
MIN_INLIER_RATIO = 0.0

# Output settings. Use 512 to match the old pipeline; set None to keep each shared crop's native size.
OUTPUT_SIZE = 512
JPEG_QUALITY = 95

print('Base:', BASE)
print('Processed images:', PROCESSED_DIR)
print('Labels CSV:', LABELS_CSV)
print('Output dir:', OUTPUT_DIR)
print('Output CSV:', OUTPUT_CSV)
assert os.path.isdir(PROCESSED_DIR), PROCESSED_DIR
assert os.path.exists(LABELS_CSV), LABELS_CSV
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Load models once. Re-run this cell if CUDA memory gets into a bad state.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

extractor = SuperPoint(max_num_keypoints=MAX_KEYPOINTS).eval().to(device)
matcher = LightGlue(features='superpoint').eval().to(device)

In [ ]:
DAYLIKE = {'daytime', 'day', 'morning'}

def pick_anchor(group):
    tod = group['time_of_day'].astype(str).str.lower().str.strip()
    weather = group['weather'].astype(str).str.lower().str.strip()
    candidates = group[tod.isin(DAYLIKE) & (weather == 'clear')]
    if candidates.empty:
        candidates = group[tod.isin(DAYLIKE)]
    if candidates.empty:
        candidates = group
    return candidates.iloc[0]

def load_bgr_checked(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return img

def lightglue_match(anchor_path, target_path):
    image0_raw = load_image(anchor_path, resize=RESIZE)
    image1_raw = load_image(target_path, resize=RESIZE)
    image0 = image0_raw.mean(dim=0, keepdim=True).unsqueeze(0).to(device)
    image1 = image1_raw.mean(dim=0, keepdim=True).unsqueeze(0).to(device)

    with torch.inference_mode():
        feats0 = extractor({'image': image0})
        feats1 = extractor({'image': image1})
        matches01 = matcher({'image0': feats0, 'image1': feats1})

    kpts0 = feats0['keypoints'][0].detach().cpu().numpy()
    kpts1 = feats1['keypoints'][0].detach().cpu().numpy()
    matches = matches01['matches'][0].detach().cpu().numpy()
    if len(matches) == 0:
        return None, {'matches': 0, 'inliers': 0, 'inlier_ratio': 0.0, 'reason': 'no_matches'}

    mkpts0 = kpts0[matches[:, 0]]
    mkpts1 = kpts1[matches[:, 1]]
    return (image0_raw, image1_raw, mkpts0, mkpts1, len(matches)), None

def estimate_target_to_anchor(anchor_bgr, target_bgr, anchor_path, target_path):
    match_result, error = lightglue_match(anchor_path, target_path)
    if error is not None:
        return None, error

    image0_raw, image1_raw, mkpts0, mkpts1, n_matches = match_result
    if n_matches < MIN_MATCHES:
        return None, {'matches': n_matches, 'inliers': 0, 'inlier_ratio': 0.0, 'reason': 'not_enough_matches'}

    h0, w0 = anchor_bgr.shape[:2]
    h1, w1 = target_bgr.shape[:2]
    scale0 = np.array([w0 / image0_raw.shape[2], h0 / image0_raw.shape[1]])
    scale1 = np.array([w1 / image1_raw.shape[2], h1 / image1_raw.shape[1]])
    mkpts0_orig = mkpts0 * scale0
    mkpts1_orig = mkpts1 * scale1

    H, inlier_mask = cv2.findHomography(mkpts1_orig, mkpts0_orig, cv2.USAC_MAGSAC, RANSAC_REPROJ_THRESHOLD)
    if H is None or inlier_mask is None:
        return None, {'matches': n_matches, 'inliers': 0, 'inlier_ratio': 0.0, 'reason': 'homography_failed'}

    inliers = int(inlier_mask.ravel().sum())
    inlier_ratio = inliers / max(n_matches, 1)
    if inliers < MIN_INLIERS:
        return None, {'matches': n_matches, 'inliers': inliers, 'inlier_ratio': inlier_ratio, 'reason': 'too_few_inliers'}
    if MIN_INLIER_RATIO > 0 and inlier_ratio < MIN_INLIER_RATIO:
        return None, {'matches': n_matches, 'inliers': inliers, 'inlier_ratio': inlier_ratio, 'reason': 'too_few_inliers'}

    return H, {
        'matches': n_matches,
        'inliers': inliers,
        'inlier_ratio': inlier_ratio,
        'reason': 'ok',
        'homography': H.tolist(),
    }

def resize_output(img):
    if OUTPUT_SIZE is None:
        return img
    return cv2.resize(img, (OUTPUT_SIZE, OUTPUT_SIZE), interpolation=cv2.INTER_AREA)

In [ ]:
def align_location_group(location, group):
    group = group.copy().reset_index(drop=True)
    anchor_row = pick_anchor(group)
    anchor_file = anchor_row['file_name']
    anchor_path = os.path.join(PROCESSED_DIR, anchor_file)

    try:
        anchor_bgr = load_bgr_checked(anchor_path)
    except FileNotFoundError:
        print(f"[SKIP] {location}: missing anchor {anchor_path}")
        return []

    h0, w0 = anchor_bgr.shape[:2]
    aligned_items = [{
        'row': anchor_row,
        'image': anchor_bgr,
        'mask': np.ones((h0, w0), dtype=np.uint8) * 255,
        'stats': {'matches': 0, 'inliers': 0, 'inlier_ratio': 1.0, 'reason': 'anchor'},
        'homography': np.eye(3),
        'is_anchor': True,
    }]

    print(f"\nProcessing {location}: {len(group)} images | anchor={anchor_file}")
    for _, row in group.iterrows():
        if row['file_name'] == anchor_file:
            continue

        target_file = row['file_name']
        target_path = os.path.join(PROCESSED_DIR, target_file)
        try:
            target_bgr = load_bgr_checked(target_path)
        except FileNotFoundError:
            print(f"  [MISS] {target_file}")
            continue

        H, stats = estimate_target_to_anchor(anchor_bgr, target_bgr, anchor_path, target_path)
        if H is None:
            print(
                f"  [FAIL] {target_file} "
                f"matches={stats['matches']} inliers={stats['inliers']} "
                f"ratio={stats['inlier_ratio']:.2f} reason={stats['reason']}"
            )
            continue

        aligned = cv2.warpPerspective(target_bgr, H, (w0, h0))
        source_mask = np.ones(target_bgr.shape[:2], dtype=np.uint8) * 255
        warped_mask = cv2.warpPerspective(source_mask, H, (w0, h0))
        aligned_items.append({
            'row': row,
            'image': aligned,
            'mask': warped_mask,
            'stats': stats,
            'homography': H,
            'is_anchor': False,
        })
        print(f"  [OK] {target_file} matches={stats['matches']} inliers={stats['inliers']} ratio={stats['inlier_ratio']:.2f}")

    shared_mask = aligned_items[0]['mask']
    for item in aligned_items[1:]:
        shared_mask = cv2.bitwise_and(shared_mask, item['mask'])
    coords = cv2.findNonZero(shared_mask)
    if coords is None:
        print(f"  [SKIP] {location}: no shared crop after alignment")
        return []

    x, y, crop_w, crop_h = cv2.boundingRect(coords)
    location_dir = os.path.join(OUTPUT_DIR, safe_name(location))
    os.makedirs(location_dir, exist_ok=True)

    records = []
    for item in aligned_items:
        row = item['row']
        cropped = resize_output(item['image'][y:y + crop_h, x:x + crop_w])
        out_name = f"{safe_name(row['file_name'])}_aligned.jpg"
        out_path = os.path.join(location_dir, out_name)
        cv2.imwrite(out_path, cropped, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])

        stats = item['stats']
        records.append({
            'location': location,
            'source_file': row['file_name'],
            'aligned_file': os.path.relpath(out_path, OUTPUT_DIR),
            'is_anchor': item['is_anchor'],
            'anchor_file': anchor_file,
            'time_of_day': row.get('time_of_day', ''),
            'weather': row.get('weather', ''),
            'matches': stats['matches'],
            'inliers': stats['inliers'],
            'inlier_ratio': round(float(stats['inlier_ratio']), 4),
            'crop_x': x,
            'crop_y': y,
            'crop_w': crop_w,
            'crop_h': crop_h,
            'output_size': OUTPUT_SIZE or '',
            'homography_json': json.dumps(item['homography'].tolist()),
        })

    print(f"  [SAVE] {len(records)} cropped aligned images | crop=({x}, {y}, {crop_w}, {crop_h})")
    return records

In [ ]:
labels_df = pd.read_csv(LABELS_CSV)
required_cols = {'file_name', 'location', 'time_of_day', 'weather'}
missing_cols = required_cols - set(labels_df.columns)
assert not missing_cols, f'Missing columns: {missing_cols}'

labels_df['file_name'] = labels_df['file_name'].astype(str)
labels_df['location'] = labels_df['location'].astype(str).str.lower().str.strip()
labels_df['time_of_day'] = labels_df['time_of_day'].astype(str).str.lower().str.strip()
labels_df['weather'] = labels_df['weather'].astype(str).str.lower().str.strip()

if LOCATION_FILTER is not None:
    wanted = {str(x).lower().strip() for x in LOCATION_FILTER}
    labels_df = labels_df[labels_df['location'].isin(wanted)].copy()

all_records = []
groups = list(labels_df.groupby('location', sort=True))
for location, group in tqdm(groups, desc='Locations'):
    all_records.extend(align_location_group(location, group))

out_df = pd.DataFrame(all_records)
out_df.to_csv(OUTPUT_CSV, index=False)
print(f'\nDone. Saved {len(out_df)} rows to {OUTPUT_CSV}')
display(out_df.head())

In [ ]:
# Quick visual check for one saved location.
if len(out_df):
    sample_location = out_df['location'].iloc[0]
    sample = out_df[out_df['location'] == sample_location].head(4)
    fig, axes = plt.subplots(1, len(sample), figsize=(5 * len(sample), 5))
    if len(sample) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, sample.iterrows()):
        img = cv2.imread(os.path.join(OUTPUT_DIR, row['aligned_file']), cv2.IMREAD_COLOR)
        ax.imshow(bgr_to_rgb(img))
        ax.set_title(row['source_file'])
        ax.axis('off')
    plt.show()
else:
    print('No aligned outputs to preview.')